# Mini-application de Mendelian Randomization

Objectif : tester si une exposition pourrait influencer le risque de SCAD.

La Mendelian Randomization utilise des variants génétiques comme instruments, un peu comme une randomisation naturelle.

## Principe

Nous testons une relation du type :

**Variant génétique → Exposition → SCAD**

La MR ne prouve pas automatiquement la causalité. Elle dépend de plusieurs hypothèses.

In [ ]:
install.packages(c('data.table', 'dplyr', 'ggplot2'), repos = 'https://cloud.r-project.org')
install.packages('TwoSampleMR', repos = c('https://mrcieu.r-universe.dev', 'https://cloud.r-project.org'))

In [ ]:
library(data.table)
library(dplyr)
library(ggplot2)
library(TwoSampleMR)

In [ ]:
#@title Choisir l'exposition
exposure_choice <- "BMI" #@param ["BMI", "Systolic blood pressure", "LDL cholesterol"]

# Remplacer TON_GITHUB_USER par ton identifiant GitHub.
github_user <- 'takiy-berrandou'
repo_name <- 'mr-apprenti-chercheur'

base_url <- paste0('https://raw.githubusercontent.com/', github_user, '/', repo_name, '/main/data/harmonised/')

file_map <- list(
  "BMI" = paste0(base_url, "bmi_scad.tsv"),
  "Systolic blood pressure" = paste0(base_url, "sbp_scad.tsv"),
  "LDL cholesterol" = paste0(base_url, "ldl_scad.tsv")

)

cat('Exposition choisie :', exposure_choice, '\n')
cat('Fichier :', file_map[[exposure_choice]], '\n')

In [ ]:
dat <- fread(file_map[[exposure_choice]])
dat <- as.data.frame(dat)

cat('Nombre total de SNPs dans le fichier :', nrow(dat), '\n')

if ('mr_keep' %in% names(dat)) {
  dat <- dat[dat$mr_keep == TRUE, ]
}

cat('Nombre de SNPs gardés pour la MR :', nrow(dat), '\n')
head(dat)

In [ ]:
mr_results <- mr(dat)
mr_results_or <- generate_odds_ratios(mr_results)
mr_results_or

In [ ]:
ivw <- mr_results_or %>%
  filter(method == 'Inverse variance weighted') %>%
  slice(1)

cat('Résultat principal\n')
cat('==================\n\n')
cat('Exposition :', exposure_choice, '\n')
cat('Outcome : SCAD\n')
cat('Nombre de SNPs :', ivw$nsnp, '\n')
cat('Odds ratio :', round(ivw$or, 3), '\n')
cat('IC 95% :', round(ivw$or_lci95, 3), '-', round(ivw$or_uci95, 3), '\n')
cat('P-value :', signif(ivw$pval, 3), '\n\n')

if (ivw$pval < 0.05 && ivw$or > 1) {
  cat("Interprétation : l'exposition pourrait augmenter le risque de SCAD.\n")
} else if (ivw$pval < 0.05 && ivw$or < 1) {
  cat("Interprétation : l'exposition pourrait diminuer le risque de SCAD.\n")
} else {
  cat("Interprétation : pas d'association causale claire dans cette analyse.\n")
}

cat('\nAttention : ce résultat dépend des hypothèses de la MR.\n')

## Scatter plot

Chaque point représente un variant génétique.

- Axe horizontal : effet du SNP sur l’exposition.
- Axe vertical : effet du même SNP sur SCAD.
- La pente correspond à l’estimation MR.

In [ ]:
scatter <- mr_scatter_plot(mr_results, dat)
scatter[[1]]

## Forest plot

Ce graphique montre les estimations variant par variant.

In [ ]:
single_snp <- mr_singlesnp(dat)
forest <- mr_forest_plot(single_snp)
forest[[1]]

## Leave-one-out

Cette analyse retire un SNP à chaque fois. Si le résultat change fortement quand un SNP est retiré, le résultat dépend beaucoup de ce SNP.

In [ ]:
loo <- mr_leaveoneout(dat)
loo_plot <- mr_leaveoneout_plot(loo)
loo_plot[[1]]

## Tests de sensibilité

Ces tests aident à discuter les limites de l’analyse, notamment l’hétérogénéité et la pléiotropie.

In [ ]:
heterogeneity <- tryCatch(mr_heterogeneity(dat), error = function(e) NULL)
heterogeneity

pleiotropy <- tryCatch(mr_pleiotropy_test(dat), error = function(e) NULL)
pleiotropy